In [19]:
# Run this cell before the simulator
%pip install azure-eventhub --break-system-packages



StatementMeta(, d6c2537c-39a4-4ae2-b925-463115fc0984, 50, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [20]:
from azure.eventhub import EventHubProducerClient, EventData
import json, time
from pyspark.sql import functions as F

StatementMeta(, d6c2537c-39a4-4ae2-b925-463115fc0984, 52, Finished, Available, Finished, False)

In [21]:
CONNECTION_STR = "Endpoint=sb://esehpnms2tzp4rnd41nn6f.servicebus.windows.net/;SharedAccessKeyName=key_345f1f78-7d82-4c1f-9dd0-efc5280ee522;SharedAccessKey=xtXLAWsN7vv0hyBY6ZiYex5BhoSYxYdeC+AEhOULaqA=;EntityPath=esehpnms2tzp4rnd41nn6f_eh"
BATCH_SIZE     = 50    # transactions per send
DELAY_SECONDS  = 1     # 1 second between batches (adjustable)

StatementMeta(, d6c2537c-39a4-4ae2-b925-463115fc0984, 53, Finished, Available, Finished, False)

In [22]:
df = spark.read.format("delta").table("silver_fraud_scores") \
         .select("Time","Amount","hour_of_day","fraud_score_pct",
                 "fraud_prob","iso_flag",
                 "fraud_tier","action_required","Class","amount_bin") \
         .orderBy("Time") \
         .limit(5000)  # stream first 5,000 for POC demo

rows = df.collect()
producer = EventHubProducerClient.from_connection_string(CONNECTION_STR)

print(f"Streaming {len(rows)} transactions...")
sent = 0
with producer:
    for i in range(0, len(rows), BATCH_SIZE):
        batch = producer.create_batch()
        for row in rows[i:i+BATCH_SIZE]:
            event = {
                "Time":             row["Time"],
                "Amount":           float(row["Amount"]),
                "hour_of_day":      int(row["hour_of_day"]),
                "fraud_score_pct":  float(row["fraud_score_pct"]),
                "xgb_fraud_prob":   float(row["fraud_prob"]),
                "iso_flag":         int(row["iso_flag"]),
                "fraud_tier":       row["fraud_tier"],
                "action_required":  row["action_required"],
                "actual_fraud":     int(row["Class"]),
                "amount_bin":       row["amount_bin"],
            }
            batch.add(EventData(json.dumps(event)))
        producer.send_batch(batch)
        sent += min(BATCH_SIZE, len(rows) - i)
        print(f"  Sent {sent}/{len(rows)} transactions")
        time.sleep(DELAY_SECONDS)

print(f"✅ Stream simulation complete. {sent} transactions sent.")

StatementMeta(, d6c2537c-39a4-4ae2-b925-463115fc0984, 54, Finished, Available, Finished, False)

Streaming 5000 transactions...
  Sent 50/5000 transactions
  Sent 100/5000 transactions
  Sent 150/5000 transactions
  Sent 200/5000 transactions
  Sent 250/5000 transactions
  Sent 300/5000 transactions
  Sent 350/5000 transactions
  Sent 400/5000 transactions
  Sent 450/5000 transactions
  Sent 500/5000 transactions
  Sent 550/5000 transactions
  Sent 600/5000 transactions
  Sent 650/5000 transactions
  Sent 700/5000 transactions
  Sent 750/5000 transactions
  Sent 800/5000 transactions
  Sent 850/5000 transactions
  Sent 900/5000 transactions
  Sent 950/5000 transactions
  Sent 1000/5000 transactions
  Sent 1050/5000 transactions
  Sent 1100/5000 transactions
  Sent 1150/5000 transactions
  Sent 1200/5000 transactions
  Sent 1250/5000 transactions
  Sent 1300/5000 transactions
  Sent 1350/5000 transactions
  Sent 1400/5000 transactions
  Sent 1450/5000 transactions
  Sent 1500/5000 transactions
  Sent 1550/5000 transactions
  Sent 1600/5000 transactions
  Sent 1650/5000 transactions